# 2장 텍스트 데이터 다루기: 따라 치기 노트북

『밑바닥부터 만들면서 배우는 LLM』 2장을 따라 치며 공부하는 노트북입니다.

**진행 방법**
1. 절마다 책을 보며 `✏️` 셀에 코드를 직접 입력하고 실행하세요. 셀은 필요한 만큼 추가하면 됩니다.
2. 출력(토큰, 토큰 ID, 텐서 shape)을 **매번 책과 비교**하세요.
3. 주석은 책 문장을 옮기지 말고 **"이 줄이 왜 필요한지"를 내 말로** 쓰세요.
4. 🔍 **확인 셀**에서 `✅`가 나오면 다음 절로 넘어가세요.
5. 장을 마치면 `notes.md`를 채우고, 핵심 두 가지(토크나이저 클래스, 슬라이딩 윈도우 데이터셋)를 보지 않고 다시 짜 보세요.

막히면 [책의 공식 코드](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/01_main-chapter-code)를 참고하세요.

## 0. 환경 설정
Colab에는 `torch`가 설치되어 있고, GPT-2의 BPE 토크나이저인 `tiktoken`만 추가로 설치합니다. 2장은 GPU가 없어도 됩니다.

In [ ]:
!pip install -q tiktoken

from importlib.metadata import version
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 데이터 불러오기
같은 폴더의 `the-verdict.txt`를 읽습니다. Colab처럼 파일이 없으면 이 저장소에서 내려받습니다.

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
if not os.path.exists(file_path):
    url = "https://raw.githubusercontent.com/uisang2004-maker/LLM-/main/ch02_tokenizer/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("총 문자 수:", len(raw_text))
print(raw_text[:99])

## 2.1 단어 임베딩 이해하기
코드 없이 개념을 읽는 절입니다. 읽고 나서 **"임베딩이란?"** 을 한 문장으로 `notes.md`에 적어 보세요.

## 2.2 텍스트 토큰화하기
정규 표현식(`re`)으로 텍스트를 단어와 구두점으로 나눕니다. 나눈 결과를 `preprocessed`에 담으세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: re.split으로 텍스트 나누기 → preprocessed

In [ ]:
# 🔍 확인
print("토큰 수:", len(preprocessed))
print(preprocessed[:30])
print("✅ 책의 토큰 수, 앞부분 출력과 같은지 비교하세요")

## 2.3 토큰을 토큰 ID로 변환하기
고유한 토큰에 번호를 매겨 어휘 사전(`vocab`)을 만들고, `encode`와 `decode`가 있는 `SimpleTokenizerV1` 클래스를 만듭니다.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: 어휘 사전 vocab 만들기

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: SimpleTokenizerV1 클래스 (encode, decode)

In [ ]:
# 🔍 확인: 문장 → ID → 문장
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print("어휘 사전 크기:", len(vocab))
print(ids)
print(tokenizer.decode(ids))
assert isinstance(ids, list) and all(isinstance(i, int) for i in ids), "encode는 정수 리스트를 돌려줘야 해요"
print("✅ encode / decode 동작 확인")

## 2.4 특수 문맥 토큰 추가하기
어휘 사전에 없는 단어를 위한 `<|unk|>`와 문서 경계를 표시하는 `<|endoftext|>`를 추가하고, `SimpleTokenizerV2`를 만듭니다.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: 특수 토큰을 추가한 어휘 사전

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: SimpleTokenizerV2 클래스

In [ ]:
# 🔍 확인: 사전에 없는 단어가 <|unk|>로 바뀌는지
tokenizer = SimpleTokenizerV2(vocab)
text = " <|endoftext|> ".join(["Hello, do you like tea?", "In the sunlit terraces of the palace."])
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))
assert "<|unk|>" in tokenizer.decode(tokenizer.encode(text)), "사전에 없는 'Hello' 등이 <|unk|>로 바뀌어야 해요"
print("✅ 특수 토큰 처리 확인")

## 2.5 바이트 페어 인코딩 (BPE)
GPT-2가 쓰는 BPE 토크나이저를 `tiktoken`으로 불러옵니다. 공부 노트의 BPE 정리와 비교해 보세요. 사전에 없는 단어도 `<|unk|>` 없이 조각으로 나뉘는 게 핵심이에요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: tiktoken으로 gpt2 토크나이저 불러와서 encode / decode

In [ ]:
# 🔍 확인: 처음 보는 단어도 조각으로 나뉘는지
import tiktoken
_bpe = tiktoken.get_encoding("gpt2")
for word in ["Akwirw ier", "someunknownPlace"]:
    _ids = _bpe.encode(word)
    print(word, "→", _ids, "→", [_bpe.decode([i]) for i in _ids])
print("어휘 사전 크기:", _bpe.n_vocab)
print("✅ BPE는 <|unk|> 없이 어떤 문자열도 표현할 수 있어요")

## 2.6 슬라이딩 윈도우로 데이터 샘플링하기 ⭐
"다음 토큰 맞히기"를 학습 데이터로 바꾸는 핵심 절이에요. **입력을 한 칸 밀면 정답**이 됩니다. `GPTDatasetV1`과 `create_dataloader_v1`을 만드세요. 문서 분류와 NER에서 만든 `Dataset`과 비교해 보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: 입력과 한 칸 밀린 정답(target) 예시 출력

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: GPTDatasetV1 클래스

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: create_dataloader_v1 함수

In [ ]:
# 🔍 확인: 정답이 입력을 한 칸 민 것인지
import torch
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("inputs:\n", inputs)
print("targets:\n", targets)
assert inputs.shape == (8, 4) and targets.shape == (8, 4), f"shape이 (8, 4)여야 해요: {tuple(inputs.shape)}"
assert torch.equal(inputs[:, 1:], targets[:, :-1]), "정답은 입력을 한 칸 왼쪽으로 민 것이어야 해요"
print("✅ 슬라이딩 윈도우 데이터셋 확인")

## 2.7 토큰 임베딩 만들기
`torch.nn.Embedding`으로 토큰 ID를 벡터로 바꿉니다. 임베딩 층은 **번호로 행을 찾아오는 표**라는 점을 확인해 보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: 작은 Embedding 층 만들어서 가중치와 결과 비교

## 2.8 단어 위치 인코딩하기
토큰 임베딩에 **위치 임베딩**을 더해 LLM의 입력을 완성합니다. 공부 노트의 "임베딩 + 위치 정보"가 바로 이것이에요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: 토큰 임베딩 + 위치 임베딩 → input_embeddings

In [ ]:
# 🔍 확인: 최종 입력의 shape
print(input_embeddings.shape)
assert tuple(input_embeddings.shape) == (8, 4, 256), "책 설정(batch 8, max_length 4, 차원 256)이면 (8, 4, 256)이어야 해요"
print("✅ (배치, 토큰 수, 임베딩 차원) = (8, 4, 256)")

---
## 2장 마무리
- [ ] `notes.md` 채우기
- [ ] 이 노트북을 닫고 `SimpleTokenizerV2`를 보지 않고 다시 짜기
- [ ] `GPTDatasetV1`과 `create_dataloader_v1`을 보지 않고 다시 짜기 (다시 짠 코드는 `.py` 파일로 저장해 보기)
- [ ] 노트북을 .ipynb로 다운로드해 이 폴더에 넣고 커밋, push